In [6]:
from com.example.agentic.agent.LLMManager import LLMManager
# 
llm = LLMManager.create_llm('ollama')
llm.call('Hello')

'How can I assist you today?'

#### LOAD Documents

In [ ]:
from com.example.agentic.loader.LoadManager import LoadManager
# Website Data Ingestion 
documents = LoadManager.from_url(["https://docs.crewai.com/how-to/Installing-CrewAI/"])

#### Chunkings

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

#
print(len(chunks))


##### Embaddings

In [ ]:
from com.example.agentic.embedding.EmbeddingManager import EmbeddingManager
from com.example.agentic.vectors.VectorStoreManager import VectorStoreManager

#chunks=[doc.page_content for doc in documents]

### Convert the text to embeddings
embedding_manager=EmbeddingManager()
embeddings = embedding_manager.embed_documents(documents)
print("[INFO] Example embedding:", embeddings[0] if len(embeddings) > 0 else None)

##store int the vector dtaabase
vectorstore = VectorStoreManager()
vectorstore.add_documents(documents,embeddings)

#### Agent Response Formatters

In [1]:
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import SerperDevTool
from pydantic import BaseModel, Field
from typing import List, Dict
from datetime import datetime

class ResearchPoint(BaseModel):
    topic: str = Field(description="The main topic or area being discussed")
    findings: str = Field(description="The key findings or insights about this topic")
    relevance: str = Field(description="Why this finding is relevant or important")
    sources: List[Dict[str, str]] = Field(
        description="source information with title and website url for each finding",
        default_factory=list
    )

class ResearchOutput(BaseModel):
    research_points: List[ResearchPoint] = Field(description="List of research findings")
    summary: str = Field(description="Brief summary of overall findings")

class ResearchImage(BaseModel):
    topic: str = Field(description="The main topic or area being discussed")
    relevance: str = Field(description="Why this image is relevant or important")
    sources: List[Dict[str, str]] = Field(
        description="List of images with source_webpage and image_url",
        default_factory=list
    )

class ResearchImageOutput(BaseModel):
    research_images: List[ResearchImage] = Field(description="List of top images on topic")
    summary: str = Field(description="Brief summary of image findings")    

class ExecutiveReportSection(BaseModel):
    section_emoji: str = Field(description="Section emoji (e.g., 🔍, 📊, 🎯)")
    section_title: str = Field(description="Section title")
    section_content: str = Field(description="Main content of the section")
    key_insights: List[str] = Field(description="Key insights from this section")
    recommendations: List[str] = Field(
        default_factory=list,
        description="Optional recommendations based on findings"
    )
    sources: List[Dict[str, str]] = Field(
        description="Sources with title and URL for this section",
        default_factory=list
    )

class ExecutiveReport(BaseModel):
    report_title: str = Field(description="Title of the report")
    generation_date: str = Field(description="Report generation date")
    executive_summary: str = Field(description="A concise executive summary")
    key_findings: List[Dict[str, str]] = Field(
        description="List of key findings with their sources",
        default_factory=list
    )
    report_sections: List[ExecutiveReportSection] = Field(
        description="Detailed report sections"
    )
    next_steps: List[str] = Field(description="Recommended next steps")
    sources: List[Dict[str, str]] = Field(
        description="All sources used in the report",
        default_factory=list
    )

#### Crew Base

In [2]:
from crewai_tools import ScrapeWebsiteTool
from crewai_tools import TavilySearchTool

from crewai.tools import tool
# Perform search for provide topic
websearch_tool = SerperDevTool()

# 1. Initialize the tool with image search enabled
tavily_tool = TavilySearchTool(
    api_key=os.environ.get("TAVILY_API_KEY"),
    include_images=True,
    max_results=10
)
# Initialize the tool to scrape the target website
@tool
def image_scrap_tool():
    """
    scrape images and content for target website
    """
    #website_url='https://www.designdocs.dev/'
    scrape_tool = ScrapeWebsiteTool()

In [21]:
from crewai.knowledge.source.text_file_knowledge_source import TextFileKnowledgeSource

text_kw_source = TextFileKnowledgeSource(
    file_paths=[
        "designs.txt", 
        "machine_learning.txt",
        "python_intro.txt"
    ]
)

from crewai.knowledge.source.json_knowledge_source import JSONKnowledgeSource
json_source = JSONKnowledgeSource(
    file_paths=["designs-research.json"]
)

##### Design Crew

In [25]:
from datetime import datetime
from crewai import Task

from crewai_tools import ScrapeWebsiteTool, SerperDevTool
#from com.example.agentic.tools.DesignSearchTool import DesignSearchTool, DesignInput


@CrewBase
class LatestDesignDevelopmentCrew():
    """LatestAiDevelopment crew"""

    agents_config = "config/agents.yaml"
    tasks_config = "config/tasks.yaml"

    @agent
    def content_researcher(self) -> Agent:
        return Agent(
            config=self.agents_config['content_researcher'],
            verbose=True,
            tools=[
                SerperDevTool(),
                ScrapeWebsiteTool()
                # DesignSearchTool()
            ],
            #knowledge_sources=[text_kw_source],
            embedder={
                "provider": "openai",
                "config": {
                    "model": "nomic-embed-text",
                    "api_key":"",
                    "platform_url":"http://localhost:11434/v1"
                },
            },
            llm=llm
        )
    
    @agent
    def website_collector(self) -> Agent:
        return Agent(
            config=self.agents_config['website_collector'],
            knowledge_sources=[json_source],
            verbose=True,
            tools=[],
            embedder={
                "provider": "openai",
                "config": {
                    "model": "nomic-embed-text",
                    "api_key":"",
                    "platform_url":"http://localhost:11434/v1"
                },
            },
            llm=llm
        )
    
    # @agent
    # def image_extractor(self) -> Agent:
    #     return Agent(
    #         config=self.agents_config['image_extractor'],
    #         verbose=True,
    #         knowledge_sources=[json_source],
    #         tools=[tavily_tool],
    #         allow_delegation=False,
    #         embedder={
    #             "provider": "openai",
    #             "config": {
    #                 "model": "nomic-embed-text",
    #                 "api_key":"",
    #                 "platform_url":"http://localhost:11434/v1"
    #             },
    #         },
    #         llm=llm
    #     )
    
    # @agent
    # def reporting_analyst(self) -> Agent:
    #     return Agent(
    #         config=self.agents_config['reporting_analyst'],
    #         verbose=True,
    #         llm=llm
    #     )
    
    # @agent
    # def formatter(self) -> Agent:
    #     return Agent(
    #         config=self.agents_config['formatter'],
    #         verbose=True,
    #         llm=llm
    #     )

    @task
    def research_task(self) -> Task:
        return Task(
            config=self.tasks_config['research_task'], 
            output_json=ResearchOutput
        )


    @task
    def collect_urls_task(self) -> Task:
        return Task(
            config=self.tasks_config['collect_urls_task'],
            context=[self.research_task()]
        )
    
    # @task
    # def find_images_task(self) -> Task:
    #     return Task(
    #         config=self.tasks_config['find_images_task'],
    #         output_json=ResearchImageOutput,
    #         output_file=f'outputs/L005/designs-research_{run_id}.json',
    #     )
    
    # @task
    # def reporting_task(self) -> Task:
    #     return Task(
    #         config=self.tasks_config['reporting_task'],
    #         output_pydantic=ExecutiveReport,
    #         output_file=f'outputs/L005/designs-report_{run_id}.json',    
    #     )

    # @task
    # def formatting_task(self) -> Task:
    #     return Task(
    #         config=self.tasks_config['formatting_task']
    #         output_file=f'outputs/L005/formatted-design-report_{run_id}.md',
    #     )

	
    @crew
    def crew(self) -> Crew:
        """Creates the LatestDesignDevelopmentCrew crew"""
        return Crew(
            agents=self.agents, # Automatically created by the @agent decorator
            tasks=self.tasks, # Automatically created by the @task decorator
            process=Process.sequential,
            verbose=True,
        )

In [26]:
from datetime import datetime
_exe_date = datetime.now().strftime('%Y-%m-%d')
_exe_id = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f" LatestDesignDevelopmentCrew triggered on {_exe_date} with execution id {_exe_id}")
def run():
    """
    Run the crew.
    """
    inputs = {
        'topic': 'Microservice Design',
        'date': _exe_date,
        'run_id': _exe_id
    }
    LatestDesignDevelopmentCrew().crew().kickoff(inputs=inputs)

 LatestDesignDevelopmentCrew triggered on 2026-04-21 with execution id 20260421_210100


In [ ]:
run()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: dc0b3016-83f4-4881-a9a5-00cb15ce59d2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: research_task                                                                                            │
│  ID: 6697b285-2204-4502-84b7-b571e175ca40                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Researcher                                                                                      │
│                                                                                                                 │
│  Task: Conduct a thorough research about Microservice Design Make sure you find minimume 10 relevant and        │
│  accurate content on solution design for Microservice Design.                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Researcher                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  research_points=[ResearchPoint(topic='Microservice Design', findings='Microservices can improve scalability    │
│  and flexibility by allowing for independent subsystems to communicate with each other.', relevance='This       │
│  finding is relevant because it provides insight into the benefits of microservices in improving                │
│  organizational flexibility and scalability.', sources=[{'title': "The book <a                                  │
│  href='https://www.amazon.com/Design-Microservices-Software-Evangelists-Carters/dp/1491932384'                  │
│  target='_blank'>Understanding Microservices</a> by John Ellis and Mark McGrath", 'type': 'book'}, {'title':    │
│  "The article <a href='https://blog.devpost.com/scale-it-up-with-everyone-online-1f35d5c88f4e/article/1216'     │
│  target='_blank'>", 'type': 'article'}]), ResearchPoint(topic='Microservice Design', findings="Designing        │
│  microservices requires attention to detail and a clear understanding of each service's functionality.",        │
│  relevance='This finding is relevant because it highlights the importance of careful design when implementing   │
│  microservices.', sources=[{'title': "The course <a href='https://www.courserae.edu/courses/BBA1116'            │
│  target='_blank'>Introduction to Microservices</a>", 'type': 'course'}]), ResearchPoint(topic='Microservice     │
│  Design', findings='One key consideration when designing microservices is ensuring consistent communication     │
│  between them.', relevance='This finding is relevant because it emphasizes the need for clear communication     │
│  among services in a microservices architecture.', sources=[{'title': "The webinar <a href='https://video       │
│  conference-devpost.com/video/scale-it-up-with-everyone-online-1f35d5c88f4e/article/1217' target='_blank'> ",   │
│  'type': 'webinar'}]), ResearchPoint(topic='Microservice Design', findings='Designing microservices can be      │
│  challenging when working with external partners or vendors.', relevance='This finding is relevant because it   │
│  highlights the need for careful planning and risk management in microservices development.',                   │
│  sources=[{'title': "The report <a                                                                              │
│  href='https://www.researchgate.net/publication/278144814-Microservices-and-Exposure-to-Risk-in-the-enterprise  │
│  ' target='_blank'>", 'type': 'report'}]), ResearchPoint(topic='Microservice Design', findings='One effective   │
│  way to improve microservices is by implementing a service registry.', relevance='This finding is relevant      │
│  because it highlights the potential of service discovery in facilitating communication among services.',       │
│  sources=[{'title': "The tutorial <a                                                                            │
│  href='https://www.tutorialspoint.com/microservices/microservices_service_registry.htm' target='_blank'>  ",    │
│  'type': 'tutorial'}]), ResearchPoint(topic='Microservice Design', findings='Designing microservices requires   │
│  a strong understanding of architecture principles.', relevance='This finding is relevant because it            │
│  emphasizes the importance of knowledge sharing and standardization in architectural design.',                  │
│  sources=[{'title': "The paper <a href='https://dl.acm.

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: research_task                                                                                            │
│  Agent: Content Researcher                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Website url collector                                                                                   │
│                                                                                                                 │
│  Task: make sure you find all web page urls from provided context.                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Website url collector                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  I'll create a single JSON object containing all the URL lists from the provided context.                       │
│                                                                                                                 │
│  ```                                                                                                            │
│  [                                                                                                              │
│    {                                                                                                            │
│      "name": "DevOps Practices",                                                                                │
│      "urls": [                                                                                                  │
│        "https://www.huffpost.com/national-news/devops-best-practices_N_16921589.html"                           │
│      ]                                                                                                          │
│    },                                                                                                           │
│    {                                                                                                            │
│      "name": "Automation of Microservices Deployment",                                                          │
│      "urls": [                                                                                                  │
│        "https://www.huffpost.com/national-news/how-to-automate-microservices-deployment_N_16921591.html"        │
│      ]                                                                                                          │
│    },                                                                                                           │
│    {                                                                                                            │
│      "name": "Service Discovery",                                                                               │
│      "urls": [                                                                                                  │
│        "https://www.huffpost.com/national-news/how-to-use-service-discovery_N_16921593.html"                    │
│      ]                                                                                                          │
│    },                                                                                                           │
│    {                                                                                                            │
│      "name": "Monitoring and Error Handling",                                                                   │
│      "urls": [                                                                                                  │
│        "https://www.huffpost.com/national-news/how-to-monitor-microservices_N_16921595.html"                    │
│      ]                                                                                                          │
│    },                                                                                                           │
│    {                                                                                                            │
│      "name": "Event-Driven Architecture",              

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: collect_urls_task                                                                                        │
│  Agent: Website url collector                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ 74deb870-1a1d-4990-92f0-288dd8fa3df6                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/74deb870-1a1d-499 │
│ 0-92f0-288dd8fa3df6?access_code=TRACE-f88d00a843                             │
│ 🔑 Access Code: TRACE-f88d00a843                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: dc0b3016-83f4-4881-a9a5-00cb15ce59d2                                                                       │
│  Final Output: I'll create a single JSON object containing all the URL lists from the provided context.         │
│                                                                                                                 │
│  ```                                                                                                            │
│  [                                                                                                              │
│    {                                                                                                            │
│      "name": "DevOps Practices",                                                                                │
│      "urls": [                                                                                                  │
│        "https://www.huffpost.com/national-news/devops-best-practices_N_16921589.html"                           │
│      ]                                                                                                          │
│    },                                                                                                           │
│    {                                                                                                            │
│      "name": "Automation of Microservices Deployment",                                                          │
│      "urls": [                                                                                                  │
│        "https://www.huffpost.com/national-news/how-to-automate-microservices-deployment_N_16921591.html"        │
│      ]                                                                                                          │
│    },                                                                                                           │
│    {                                                                                                            │
│      "name": "Service Discovery",                                                                               │
│      "urls": [                                                                                                  │
│        "https://www.huffpost.com/national-news/how-to-use-service-discovery_N_16921593.html"                    │
│      ]                                                                                                          │
│    },                                                                                                           │
│    {                                                                                                            │
│      "name": "Monitoring and Error Handling",                                                                   │
│      "urls": [                                                                                                  │
│        "https://www.huffpost.com/national-news/how-to-monitor-microservices_N_16921595.html"                    │
│      ]                                                                                                          │
│    },                                                                                                           │
│    {                                                                                                            │
│      "name": "Event-Driven Architecture",             

####
####